# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
First, load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display general information
meta = dataset.metadata
print(f"Name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Version: {meta.version}")
print(f"Published: {meta.datePublished}")
print(f"Keywords: {', '.join(meta.keywords)}")

## 2. Data Overview
Review available record sets and fields. All `mlcroissant` entities (record sets, fields, columns) are referenced by their `@id`, which can be found in the dataset metadata.

In [ ]:
# List all record sets and their @id
print("Record Sets available in dataset (by @id):\n")
for rs in dataset.metadata.recordSets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', rs['@id'])}")

# Example: For the main protein data table, let's search for relevant record set
protein_rs_id = None
for rs in dataset.metadata.recordSets:
    if ('protein' in rs.get('name', '').lower()) or ('Protein' in rs['@id']) or ('table' in rs.get('name', '').lower()):
        protein_rs_id = rs['@id']
        break
# Fallback to pick first if not found
if protein_rs_id is None:
    protein_rs_id = dataset.metadata.recordSets[0]['@id']
print(f"\nSelected record set for further exploration: {protein_rs_id}\n")

# List fields for the selected record set
fields = None
for rs in dataset.metadata.recordSets:
    if rs['@id'] == protein_rs_id:
        fields = rs['fields']
        break
if fields is None:
    print("No fields found for selected record set.")
else:
    print(f"Fields (@id and name) in record set '{protein_rs_id}':")
    for field in fields:
        print(f"  @id: {field['@id']}, name: {field.get('name', field['@id'])}, dataType: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis using the chosen record set and field `@id`s from the overview.

In [ ]:
# List all record set @ids
record_sets = [rs['@id'] for rs in dataset.metadata.recordSets]
dataframes = {}

for record_set_id in record_sets:
    # This will stream all records for the record set
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set: {record_set_id}")
        else:
            print(f"Record set {record_set_id} is empty.")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Pick the main protein record set for further analysis
main_rs_id = protein_rs_id
df = dataframes.get(main_rs_id)
if df is not None:
    print(f"Columns in DataFrame for '{main_rs_id}':")
    print(df.columns.tolist())
    display(df.head())
else:
    print(f"No data for record set: {main_rs_id}")

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group by key attributes. For this, select by `@id` one numeric field and one grouping field from the overview above. All fields are referenced by their `@id`.

In [ ]:
# Choose numeric and grouping field from the previous fields listing by @id
# Example field @id values (replace as appropriate for this dataset):
numeric_field_id = None
group_field_id = None
# Scan columns to propose likely numeric/group fields
if df is not None:
    numeric_candidates = [c for c in df.columns if ('abundance' in c.lower()) or ('count' in c.lower()) or ('weight' in c.lower()) or ('coverage' in c.lower()) or (df[c].dtype in [int, float])]
    group_candidates = [c for c in df.columns if ('accession' in c.lower()) or ('group' in c.lower()) or ('condition' in c.lower()) or ('sample' in c.lower())]
    # Fallbacks
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    if group_candidates:
        group_field_id = group_candidates[0]
    if not numeric_field_id:
        numeric_field_id = df.columns[0]

    print(f"Selected numeric field (@id): {numeric_field_id}")
    print(f"Selected group field (@id): {group_field_id}")

    # Safe handling for missing or non-numeric data
    filtered_df = df.copy()
    # Convert to numeric for field if possible
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    threshold = filtered_df[numeric_field_id].mean() if filtered_df[numeric_field_id].notnull().any() else 0

    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group field if it exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the chosen group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if df is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,4))
        # Show mean and errorbar if enough groups
        sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df, estimator='mean')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
We have loaded mass spectrometry data about extracellular vesicles, previewed available record sets and fields by their `@id`, extracted records, performed simple outlier filtering and normalization on a selected numeric field, and visualized the data distribution and group means. This template shows how to systemically explore any Croissant-compliant dataset using only the entity `@id` references, supporting robust and reusable scientific data workflows.